# Day 38: Prefix Caching with Hash-Based Deduplication

**Layer:** Implementation | **Prerequisite:** Previous days


## Concept Overview

Implements a production prefix cache that stores block-aligned KV states keyed by SHA-256 hash. Finds the longest cached prefix for each incoming request, saving all recomputation for matched tokens. Uses LRU eviction to manage cache capacity.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Prefix Cache with Hash-Based Deduplication


In [ ]:
import hashlib
from collections import OrderedDict

class PrefixCacheStore:
    def __init__(self, max_size=500):
        self.cache = OrderedDict()  # hash -> (tokens, kv_data)
        self.max_size = max_size
        self.hits = 0
        self.misses = 0
        self.total_tokens_saved = 0

    def _hash(self, tokens):
        return hashlib.sha256(bytes(tokens)).hexdigest()[:16]

    def get(self, tokens, block_size=16):
        """Find longest cached prefix. Returns (prefix_len, cached_data)."""
        # Check full prefix, then progressively shorter
        for end in range(len(tokens), 0, -block_size):
            # Only check block-aligned prefixes
            prefix = tokens[:end - (end % block_size)] if end % block_size else tokens[:end]
            if not prefix:
                continue
            h = self._hash(prefix)
            if h in self.cache:
                self.cache.move_to_end(h)
                self.hits += 1
                self.total_tokens_saved += len(prefix)
                return len(prefix), self.cache[h][1]
        self.misses += 1
        return 0, None

    def put(self, tokens, kv_data, block_size=16):
        """Cache all block-aligned prefixes."""
        for end in range(block_size, len(tokens)+1, block_size):
            prefix = tokens[:end]
            h = self._hash(prefix)
            if h not in self.cache:
                if len(self.cache) >= self.max_size:
                    self.cache.popitem(last=False)
                self.cache[h] = (prefix, f'kv_data_{h[:4]}')

cache = PrefixCacheStore(max_size=1000)
system_prompt = list(range(256))  # 256 token system prompt
np.random.seed(42)

# Simulate 1000 requests
for i in range(1000):
    if np.random.random() < 0.8:
        tokens = system_prompt + list(np.random.randint(256, 5000, np.random.randint(10, 100)))
    else:
        tokens = list(np.random.randint(0, 5000, np.random.randint(20, 150)))
    cached_len, _ = cache.get(tokens)
    cache.put(tokens, None)

print(f'Prefix Cache Statistics (1000 requests):')
print(f'  Hits: {cache.hits}, Misses: {cache.misses}')
print(f'  Hit rate: {cache.hits/(cache.hits+cache.misses)*100:.1f}%')
print(f'  Tokens saved (not recomputed): {cache.total_tokens_saved:,}')
print(f'  Mean tokens saved per hit: {cache.total_tokens_saved/max(cache.hits,1):.0f}')


## 2. Hash Collision Analysis


In [ ]:
# Verify that our 64-bit hash is collision-resistant for typical workloads
def estimate_collision_probability(n_entries, hash_bits=64):
    """Birthday paradox: probability of collision in n_entries with hash_bits."""
    # P(collision) ≈ 1 - e^(-n²/(2*2^b))
    import math
    exponent = -(n_entries**2) / (2 * 2**hash_bits)
    return 1 - math.exp(exponent)

print('Hash collision probability:')
for n in [1000, 10000, 100000, 1000000]:
    p = estimate_collision_probability(n, hash_bits=64)
    print(f'  {n:>8} entries: {p:.2e} probability')

print('\nWith 128-bit hash (SHA-256 first 16 bytes):')
for n in [1000, 10000, 100000, 1000000]:
    p = estimate_collision_probability(n, hash_bits=128)
    print(f'  {n:>8} entries: {p:.2e} probability')


## Experiments

1. Implement an LFU (Least Frequently Used) eviction policy and compare hit rates against LRU on a Zipf-distributed workload.
2. Extend to hierarchical caching: fast local SRAM cache → slower DRAM cache → disk cache. Measure hit rates at each level.
3. Implement a write-through vs write-back cache policy and compare consistency guarantees.


## Key Takeaways

- See concept overview above for the key points.
- Day 38 complete.
